# Mt. Rainier DAS Cross-Correlation Analysis

This notebook demonstrates the use of DAS-NoisePy for cross-correlation analysis with flexible channel pair selection, inspired by the Mt. Rainier DAS dataset analysis.

Key features:
- Flexible channel pair selection strategies
- DAS-specific preprocessing
- Cross-correlation computation with various options
- Quality control and stacking methods
- Visualization of results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Import DAS-NoisePy components
from das_noisepy import (
    DASProcessor, 
    ChannelPairSelector, 
    CrossCorrelator,
    load_config,
    create_synthetic_das_data
)

# Set up plotting
plt.style.use('default')
%matplotlib inline

## 1. Configuration and Data Loading

First, let's set up the configuration and load DAS data. For this example, we'll create synthetic data similar to what might be observed at Mt. Rainier.

In [ ]:
# Create configuration for Mt. Rainier-like DAS setup
config = {
    'das': {
        'sampling_rate': 1000.0,    # Hz
        'channel_spacing': 2.0,     # m (typical for DAS)
        'gauge_length': 20.0,       # m
    },
    'preprocessing': {
        'detrend': True,
        'taper': True,
        'taper_fraction': 0.05,
        'bandpass_filter': True,
        'freq_min': 0.5,            # Hz - focus on microseismic signals
        'freq_max': 25.0,           # Hz
        'quality_control': True,
        'snr_threshold': 2.0,
    },
    'cross_correlation': {
        'method': 'fft',
        'max_lag_time': 20.0,       # seconds - longer for regional distances
        'normalization': 'cross',
        'whitening': True,
        'whitening_freqs': [1.0, 20.0],  # Hz
        'time_window_length': 1800, # 30 minutes
        'time_window_overlap': 0.5,
        'stack_method': 'linear',
        'cc_threshold': 0.01,
    }
}

print("Configuration for Mt. Rainier DAS analysis:")
print(f"- Sampling rate: {config['das']['sampling_rate']} Hz")
print(f"- Channel spacing: {config['das']['channel_spacing']} m")
print(f"- Frequency band: {config['preprocessing']['freq_min']}-{config['preprocessing']['freq_max']} Hz")
print(f"- Max lag time: {config['cross_correlation']['max_lag_time']} s")

In [ ]:
# Create synthetic DAS data representing Mt. Rainier deployment
# Typical parameters for a volcanic DAS deployment
n_time = 60000      # 60 seconds at 1000 Hz
n_channels = 2000   # 4 km array with 2m spacing
sampling_rate = config['das']['sampling_rate']
channel_spacing = config['das']['channel_spacing']

print(f"Creating synthetic DAS data:")
print(f"- Duration: {n_time/sampling_rate:.1f} seconds")
print(f"- Array length: {n_channels * channel_spacing / 1000:.1f} km")
print(f"- {n_channels} channels")

das_data = create_synthetic_das_data(
    n_time=n_time,
    n_channels=n_channels,
    sampling_rate=sampling_rate,
    channel_spacing=channel_spacing,
    noise_level=0.5,
    signal_velocity=3000.0,  # m/s - P-wave velocity
    source_location=1000.0   # m - source at 1 km
)

print(f"\nData loaded: {das_data.strain.shape} (time, channel)")
print(f"Time range: {das_data.time[0].values:.2f} to {das_data.time[-1].values:.2f} s")
print(f"Distance range: {das_data.distance[0].values:.0f} to {das_data.distance[-1].values:.0f} m")

## 2. Data Visualization

Let's visualize the raw DAS data to understand the signal characteristics.

In [ ]:
# Plot raw DAS data
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Time series for selected channels
selected_channels = [100, 500, 1000, 1500]
for i, ch in enumerate(selected_channels):
    if i < 4:
        color = plt.cm.tab10(i)
        axes[0, 0].plot(das_data.time, das_data.strain.isel(channel=ch), 
                       label=f'Ch {ch} ({das_data.distance[ch].values:.0f}m)', 
                       color=color, alpha=0.7)

axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Strain')
axes[0, 0].set_title('DAS Time Series - Selected Channels')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Waterfall plot (subset of data and channels)
time_subset = slice(int(1.5*sampling_rate), int(3.5*sampling_rate))  # 2 seconds around signal
channel_subset = slice(400, 600, 5)  # Every 5th channel in a 200-channel window

waterfall_data = das_data.strain[time_subset, channel_subset].values
time_sub = das_data.time[time_subset].values
dist_sub = das_data.distance[channel_subset].values

# Normalize for plotting
waterfall_data = waterfall_data / np.std(waterfall_data, axis=0, keepdims=True)

im = axes[0, 1].imshow(waterfall_data.T, aspect='auto', cmap='seismic',
                      extent=[time_sub[0], time_sub[-1], dist_sub[-1], dist_sub[0]],
                      vmin=-3, vmax=3)
axes[0, 1].set_xlabel('Time (s)')
axes[0, 1].set_ylabel('Distance (m)')
axes[0, 1].set_title('DAS Waterfall Plot (Normalized)')
plt.colorbar(im, ax=axes[0, 1], label='Normalized Strain')

# Power spectral density
from scipy.signal import welch

# Calculate PSD for middle channel
mid_channel = n_channels // 2
freqs, psd = welch(das_data.strain.isel(channel=mid_channel).values, 
                  fs=sampling_rate, nperseg=2048)

axes[1, 0].loglog(freqs[1:], psd[1:])  # Skip DC component
axes[1, 0].axvspan(config['preprocessing']['freq_min'], 
                   config['preprocessing']['freq_max'], 
                   alpha=0.2, color='red', label='Processing band')
axes[1, 0].set_xlabel('Frequency (Hz)')
axes[1, 0].set_ylabel('Power Spectral Density')
axes[1, 0].set_title(f'PSD - Channel {mid_channel}')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Channel quality (simple SNR estimate)
signal_power = np.var(das_data.strain.values, axis=0)
# Use high-frequency content as noise proxy
from scipy.signal import butter, filtfilt
b, a = butter(4, 50.0/(sampling_rate/2), btype='high')
noise_data = np.zeros_like(das_data.strain.values)
for i in range(n_channels):
    noise_data[:, i] = filtfilt(b, a, das_data.strain.values[:, i])
noise_power = np.var(noise_data, axis=0)
snr_estimate = signal_power / (noise_power + 1e-12)

axes[1, 1].plot(das_data.distance, 10*np.log10(snr_estimate))
axes[1, 1].axhline(10*np.log10(config['preprocessing']['snr_threshold']), 
                   color='red', linestyle='--', label='SNR threshold')
axes[1, 1].set_xlabel('Distance (m)')
axes[1, 1].set_ylabel('SNR (dB)')
axes[1, 1].set_title('Channel Quality (Estimated SNR)')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 3. DAS Data Preprocessing

Now let's preprocess the DAS data using the DAS-NoisePy preprocessing pipeline.

In [ ]:
# Initialize DAS processor
processor = DASProcessor(config['preprocessing'])

# Set the data (normally would load from file)
processor.data = das_data

# Display data info
info = processor.get_channel_info()
print("DAS Data Information:")
for key, value in info.items():
    print(f"  {key}: {value}")

# Preprocess the data
print("\nPreprocessing DAS data...")
processed_data = processor.preprocess()

print("Preprocessing completed!")
print(f"Good channels: {np.sum(processed_data.attrs['good_channels'])}/{len(processed_data.channel)}")

In [ ]:
# Compare raw and processed data
fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# Time series comparison
ch_example = 1000
time_range = slice(int(1.5*sampling_rate), int(2.5*sampling_rate))

axes[0, 0].plot(das_data.time[time_range], das_data.strain[time_range, ch_example], 
               'b-', alpha=0.7, label='Raw')
axes[0, 0].plot(processed_data.time[time_range], processed_data.strain[time_range, ch_example], 
               'r-', alpha=0.7, label='Processed')
axes[0, 0].set_xlabel('Time (s)')
axes[0, 0].set_ylabel('Strain')
axes[0, 0].set_title(f'Channel {ch_example} - Raw vs Processed')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Frequency domain comparison
freqs_raw, psd_raw = welch(das_data.strain.isel(channel=ch_example).values, 
                          fs=sampling_rate, nperseg=2048)
freqs_proc, psd_proc = welch(processed_data.strain.isel(channel=ch_example).values, 
                           fs=sampling_rate, nperseg=2048)

axes[0, 1].loglog(freqs_raw[1:], psd_raw[1:], 'b-', alpha=0.7, label='Raw')
axes[0, 1].loglog(freqs_proc[1:], psd_proc[1:], 'r-', alpha=0.7, label='Processed')
axes[0, 1].axvspan(config['preprocessing']['freq_min'], 
                   config['preprocessing']['freq_max'], 
                   alpha=0.2, color='green', label='Processing band')
axes[0, 1].set_xlabel('Frequency (Hz)')
axes[0, 1].set_ylabel('Power Spectral Density')
axes[0, 1].set_title('PSD Comparison')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# SNR comparison
snr_processed = processed_data.attrs['snr']
good_channels = processed_data.attrs['good_channels']

axes[1, 0].plot(processed_data.distance, 10*np.log10(snr_processed), 'g-', alpha=0.7)
axes[1, 0].scatter(processed_data.distance[~good_channels], 
                   10*np.log10(snr_processed[~good_channels]), 
                   c='red', s=10, alpha=0.5, label='Bad channels')
axes[1, 0].axhline(10*np.log10(config['preprocessing']['snr_threshold']), 
                   color='red', linestyle='--', label='Threshold')
axes[1, 0].set_xlabel('Distance (m)')
axes[1, 0].set_ylabel('SNR (dB)')
axes[1, 0].set_title('Channel Quality After Processing')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Good channel distribution
axes[1, 1].hist(processed_data.distance[good_channels], bins=50, alpha=0.7, 
               color='green', label='Good channels')
axes[1, 1].hist(processed_data.distance[~good_channels], bins=50, alpha=0.7, 
               color='red', label='Bad channels')
axes[1, 1].set_xlabel('Distance (m)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Spatial Distribution of Channel Quality')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Flexible Channel Pair Selection

This is the key improvement from the Mt. Rainier notebook - flexible channel pair selection strategies.

In [ ]:
# Initialize channel pair selector
pair_selector = ChannelPairSelector(
    channel_spacing=config['das']['channel_spacing']
)

# Demonstrate different selection strategies
n_channels = len(processed_data.channel)
snr = processed_data.attrs['snr']

print("Demonstrating flexible channel pair selection strategies:")

# Strategy 1: Distance-based selection
print("\n1. Distance-based selection:")
pairs_distance = pair_selector.select_pairs_by_distance(
    n_channels=n_channels,
    min_separation=50.0,   # 50 m minimum
    max_separation=500.0,  # 500 m maximum
    step_size=50.0,        # Every 50 m
    max_pairs_per_separation=20  # Limit pairs per separation
)
print(f"Selected {len(pairs_distance)} pairs")

# Strategy 2: Quality-based selection
print("\n2. Quality-based selection:")
pairs_quality = pair_selector.select_pairs_by_quality(
    n_channels=n_channels,
    snr=snr,
    min_snr=3.0,           # Higher SNR threshold
    min_separation=50.0,
    max_separation=500.0
)
print(f"Selected {len(pairs_quality)} pairs")

# Strategy 3: Adaptive selection
print("\n3. Adaptive selection:")
pairs_adaptive = pair_selector.select_pairs_adaptive(
    n_channels=n_channels,
    snr=snr,
    target_pairs_per_distance=15,  # Target pairs per distance bin
    distance_bins=10               # Number of distance bins
)
print(f"Selected {len(pairs_adaptive)} pairs")

# Strategy 4: Custom selection (example: focus on specific array sections)
print("\n4. Custom selection - Focus on high-quality central section:")
def custom_selection(ch1, ch2, metadata):
    """Custom selection function focusing on central high-quality section"""
    # Focus on central 1 km of the array
    center_start = n_channels // 2 - 250  # 500 m before center
    center_end = n_channels // 2 + 250    # 500 m after center
    
    # Both channels must be in central section
    in_center = (center_start <= ch1 <= center_end) and (center_start <= ch2 <= center_end)
    
    # Separation constraints
    separation = abs(ch2 - ch1) * config['das']['channel_spacing']
    good_separation = 20.0 <= separation <= 200.0
    
    # Quality constraints
    good_quality = (snr[ch1] > 2.5) and (snr[ch2] > 2.5)
    
    return in_center and good_separation and good_quality

pairs_custom = pair_selector.select_pairs_custom(
    n_channels=n_channels,
    selection_func=custom_selection
)
print(f"Selected {len(pairs_custom)} pairs")

# Use adaptive selection for the analysis
selected_pairs = pairs_adaptive
print(f"\nUsing adaptive selection with {len(selected_pairs)} pairs for cross-correlation analysis")

In [ ]:
# Visualize the selected channel pairs
pair_selector.pairs = selected_pairs
fig = pair_selector.plot_pair_distribution(figsize=(15, 10))
plt.suptitle('Channel Pair Distribution - Adaptive Selection', fontsize=16, y=1.02)
plt.show()

## 5. Cross-Correlation Analysis

Now we'll compute cross-correlations for the selected channel pairs using the Mt. Rainier-inspired processing workflow.

In [ ]:
# Initialize cross-correlator
correlator = CrossCorrelator(config['cross_correlation'])

# Compute cross-correlations
print(f"Computing cross-correlations for {len(selected_pairs)} channel pairs...")
print("This may take a few moments...")

# Use time chunking for realistic processing
cc_results = correlator.compute_correlations(
    data=processed_data,
    pairs=selected_pairs,
    time_chunked=True
)

print(f"Cross-correlation computation completed!")
print(f"Results contain {len(cc_results['correlations'])} correlation pairs")
print(f"Number of time windows: {cc_results['n_windows']}")
print(f"Lag time range: {cc_results['lag_times'][0]:.2f} to {cc_results['lag_times'][-1]:.2f} s")

In [ ]:
# Stack the cross-correlations
print("Stacking cross-correlations...")
stacked_results = correlator.stack_correlations(cc_results, method='linear')

print(f"Stacking completed!")
print(f"Stacked results contain {len(stacked_results['correlations'])} correlation pairs")

# Get some statistics
qualities = [pair_data['quality'] for pair_data in stacked_results['correlations'].values()]
n_windows = [pair_data['n_windows'] for pair_data in stacked_results['correlations'].values()]

print(f"Quality range: {np.min(qualities):.4f} to {np.max(qualities):.4f}")
print(f"Windows per pair: {np.min(n_windows)} to {np.max(n_windows)}")

## 6. Results Visualization

Let's visualize the cross-correlation results to understand the signal propagation characteristics.

In [ ]:
# Extract results for plotting
lag_times = stacked_results['lag_times']
correlations = stacked_results['correlations']

# Create arrays for analysis
pair_ids = list(correlations.keys())
separations = [correlations[pid]['pair'].separation for pid in pair_ids]
distances = [correlations[pid]['pair'].distance for pid in pair_ids]
cc_data = np.array([correlations[pid]['correlation'] for pid in pair_ids])
qualities = [correlations[pid]['quality'] for pid in pair_ids]

# Sort by separation for better visualization
sort_idx = np.argsort(separations)
separations_sorted = np.array(separations)[sort_idx]
distances_sorted = np.array(distances)[sort_idx]
cc_data_sorted = cc_data[sort_idx, :]
qualities_sorted = np.array(qualities)[sort_idx]

print(f"Visualization data prepared:")
print(f"- {len(pair_ids)} correlation pairs")
print(f"- Separation range: {np.min(separations):.1f} to {np.max(separations):.1f} m")
print(f"- Distance range: {np.min(distances):.1f} to {np.max(distances):.1f} m")

In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Cross-correlation matrix (sorted by separation)
im1 = axes[0, 0].imshow(cc_data_sorted, aspect='auto', cmap='seismic',
                       extent=[lag_times[0], lag_times[-1], 
                              len(separations_sorted)-0.5, -0.5],
                       vmin=-np.percentile(np.abs(cc_data_sorted), 95),
                       vmax=np.percentile(np.abs(cc_data_sorted), 95))
axes[0, 0].set_xlabel('Lag Time (s)')
axes[0, 0].set_ylabel('Pair Index (sorted by separation)')
axes[0, 0].set_title('Cross-Correlation Matrix')
axes[0, 0].axvline(0, color='black', linestyle='--', alpha=0.5)
plt.colorbar(im1, ax=axes[0, 0], label='Correlation Coefficient')

# 2. Separation vs lag time (2D histogram of max correlation times)
max_cc_times = []
max_cc_values = []
for i, cc in enumerate(cc_data_sorted):
    max_idx = np.argmax(np.abs(cc))
    max_cc_times.append(lag_times[max_idx])
    max_cc_values.append(cc[max_idx])

scatter = axes[0, 1].scatter(separations_sorted, max_cc_times, 
                           c=np.abs(max_cc_values), cmap='viridis',
                           s=30, alpha=0.7)
axes[0, 1].set_xlabel('Separation Distance (m)')
axes[0, 1].set_ylabel('Time of Max Correlation (s)')
axes[0, 1].set_title('Apparent Velocity Analysis')
axes[0, 1].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[0, 1], label='|Correlation Coefficient|')

# Add apparent velocity lines
velocities = [1000, 2000, 3000, 4000]  # m/s
sep_range = np.linspace(np.min(separations_sorted), np.max(separations_sorted), 100)
for vel in velocities:
    travel_times = sep_range / vel
    axes[0, 1].plot(sep_range, travel_times, '--', alpha=0.7, 
                   label=f'{vel} m/s')
    axes[0, 1].plot(sep_range, -travel_times, '--', alpha=0.7)
axes[0, 1].legend()

# 3. Example cross-correlations for different separations
example_separations = [50, 100, 200, 300, 400]
colors = plt.cm.plasma(np.linspace(0, 1, len(example_separations)))

for i, target_sep in enumerate(example_separations):
    # Find closest separation
    sep_diff = np.abs(separations_sorted - target_sep)
    closest_idx = np.argmin(sep_diff)
    actual_sep = separations_sorted[closest_idx]
    
    cc = cc_data_sorted[closest_idx, :]
    axes[0, 2].plot(lag_times, cc, color=colors[i], alpha=0.8,
                   label=f'{actual_sep:.0f} m')

axes[0, 2].set_xlabel('Lag Time (s)')
axes[0, 2].set_ylabel('Correlation Coefficient')
axes[0, 2].set_title('Example Cross-Correlations')
axes[0, 2].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].legend(title='Separation')

# 4. Quality vs separation
axes[1, 0].scatter(separations_sorted, qualities_sorted, alpha=0.6, s=20)
# Add trend line
z = np.polyfit(separations_sorted, qualities_sorted, 1)
p = np.poly1d(z)
axes[1, 0].plot(separations_sorted, p(separations_sorted), "r--", alpha=0.8)
axes[1, 0].set_xlabel('Separation Distance (m)')
axes[1, 0].set_ylabel('Correlation Quality')
axes[1, 0].set_title('Quality vs Separation Distance')
axes[1, 0].grid(True, alpha=0.3)

# 5. Histogram of correlation qualities
axes[1, 1].hist(qualities_sorted, bins=30, alpha=0.7, edgecolor='black')
axes[1, 1].axvline(np.mean(qualities_sorted), color='red', linestyle='--', 
                  label=f'Mean: {np.mean(qualities_sorted):.4f}')
axes[1, 1].set_xlabel('Correlation Quality')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Distribution of Correlation Qualities')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6. Spatial distribution of correlation quality
im2 = axes[1, 2].scatter(distances_sorted, separations_sorted, 
                        c=qualities_sorted, cmap='viridis', s=20, alpha=0.7)
axes[1, 2].set_xlabel('Average Position (m)')
axes[1, 2].set_ylabel('Separation Distance (m)')
axes[1, 2].set_title('Spatial Quality Distribution')
plt.colorbar(im2, ax=axes[1, 2], label='Correlation Quality')

plt.tight_layout()
plt.show()

## 7. Advanced Analysis: Velocity Estimation

Extract velocity information from the cross-correlation results.

In [ ]:
# Velocity analysis
def estimate_velocities(separations, lag_times, cc_data, min_separation=50.0):
    """Estimate apparent velocities from cross-correlation data"""
    velocities = []
    qualities = []
    
    for i, (sep, cc) in enumerate(zip(separations, cc_data)):
        if sep < min_separation:
            continue
            
        # Find peak correlation (positive and negative lags)
        abs_cc = np.abs(cc)
        max_idx = np.argmax(abs_cc)
        max_lag = lag_times[max_idx]
        max_quality = abs_cc[max_idx]
        
        if max_lag != 0:
            apparent_velocity = sep / abs(max_lag)
            velocities.append(apparent_velocity)
            qualities.append(max_quality)
        
    return np.array(velocities), np.array(qualities)

# Estimate velocities
velocities, vel_qualities = estimate_velocities(
    separations_sorted, lag_times, cc_data_sorted
)

print(f"Velocity Analysis Results:")
print(f"- {len(velocities)} velocity estimates")
print(f"- Velocity range: {np.min(velocities):.0f} to {np.max(velocities):.0f} m/s")
print(f"- Median velocity: {np.median(velocities):.0f} m/s")
print(f"- Mean velocity: {np.mean(velocities):.0f} m/s")

# Plot velocity analysis
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Velocity histogram
axes[0].hist(velocities, bins=30, alpha=0.7, edgecolor='black', weights=vel_qualities)
axes[0].axvline(np.median(velocities), color='red', linestyle='--', 
               label=f'Median: {np.median(velocities):.0f} m/s')
axes[0].axvline(np.mean(velocities), color='orange', linestyle='--', 
               label=f'Mean: {np.mean(velocities):.0f} m/s')
axes[0].set_xlabel('Apparent Velocity (m/s)')
axes[0].set_ylabel('Quality-weighted Count')
axes[0].set_title('Distribution of Apparent Velocities')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Quality vs velocity
scatter = axes[1].scatter(velocities, vel_qualities, alpha=0.6, s=20)
axes[1].set_xlabel('Apparent Velocity (m/s)')
axes[1].set_ylabel('Correlation Quality')
axes[1].set_title('Quality vs Velocity')
axes[1].grid(True, alpha=0.3)

# Velocity vs separation (to check for bias)
valid_seps = separations_sorted[:len(velocities)]
axes[2].scatter(valid_seps, velocities, c=vel_qualities, 
               cmap='viridis', alpha=0.7, s=20)
axes[2].set_xlabel('Separation Distance (m)')
axes[2].set_ylabel('Apparent Velocity (m/s)')
axes[2].set_title('Velocity vs Separation')
plt.colorbar(axes[2].collections[0], ax=axes[2], label='Quality')

plt.tight_layout()
plt.show()

## 8. Summary and Conclusions

This notebook demonstrates the key features of DAS-NoisePy for Mt. Rainier-style cross-correlation analysis:

In [ ]:
print("DAS-NoisePy Mt. Rainier Analysis Summary")
print("=" * 50)

print(f"\nData Characteristics:")
print(f"- Array length: {n_channels * channel_spacing / 1000:.1f} km")
print(f"- Channel spacing: {channel_spacing} m")
print(f"- Sampling rate: {sampling_rate} Hz")
print(f"- Recording duration: {n_time/sampling_rate:.1f} seconds")

print(f"\nProcessing Results:")
print(f"- Good channels: {np.sum(processed_data.attrs['good_channels'])}/{n_channels} ({100*np.sum(processed_data.attrs['good_channels'])/n_channels:.1f}%)")
print(f"- Channel pairs analyzed: {len(selected_pairs)}")
print(f"- Successful correlations: {len(stacked_results['correlations'])}")
print(f"- Time windows per correlation: {np.mean(n_windows):.1f} ± {np.std(n_windows):.1f}")

print(f"\nVelocity Analysis:")
print(f"- Apparent velocity estimates: {len(velocities)}")
print(f"- Velocity range: {np.min(velocities):.0f} - {np.max(velocities):.0f} m/s")
print(f"- Median velocity: {np.median(velocities):.0f} ± {np.std(velocities):.0f} m/s")
print(f"- Expected P-wave velocity: {das_data.attrs['signal_velocity']:.0f} m/s")

print(f"\nKey Improvements over Original NoisePy:")
print(f"- ✓ Flexible channel pair selection (4 different strategies)")
print(f"- ✓ DAS-specific preprocessing and quality control")
print(f"- ✓ Adaptive pair selection for uniform spatial sampling")
print(f"- ✓ Custom selection functions for specific analysis needs")
print(f"- ✓ Quality-weighted analysis and visualization")
print(f"- ✓ Comprehensive cross-correlation visualization suite")

print(f"\nRecommendations for Real Data:")
print(f"- Adjust frequency bands based on signal characteristics")
print(f"- Use quality-based or adaptive pair selection for noisy data")
print(f"- Consider longer time windows for ambient noise studies")
print(f"- Implement phase-weighted stacking for better SNR")
print(f"- Validate velocity estimates with independent measurements")